# 03 · Una cifra correcta puede conducir a una conclusión incorrecta (47–62 min)

Primero calcula a mano: 6 kW × 10/60 h = 1 kWh. Con diez piezas correctas: 0,1 kWh/pieza. Las unidades importan: sumar kW no produce kWh.

In [ ]:
import json
import pandas as pd
from industrialdata.common import root
from industrialdata.catalog import load_catalog
from industrialdata.ingest import ingest, business_tables
folder = root()
catalog = load_catalog(folder)  # Consulta PostgreSQL: NO el catálogo JSON de pruebas.
from industrialdata.analyze import analyze
from industrialdata.reglas import energia, intensidad
telemetry, _, _, _ = ingest(folder / "raw", catalog, "exploracion-mini")
production, maintenance = business_tables(folder / "raw", catalog)
_, quality, metrics = analyze(telemetry, production, maintenance, catalog)
display(metrics)
a, b = metrics.set_index("machine_id").loc[["M01", "M02"]].to_dict("records")
assert abs(a["energy_kwh"] - 1) < 1e-9
assert abs(b["energy_kwh"] - .35) < 1e-9
assert abs(b["energy_coverage"] - .7) < 1e-9
assert pd.isna(b["kwh_per_good_unit"])

## Mira los segmentos de M02
Faltan las muestras 08:04 y 08:05. El salto de 08:03 a 08:06 dura tres minutos y supera 1,5 cadencias. Se excluye completo. Quedan siete minutos cubiertos, no ocho. La muestra de 08:10 permite cerrar el último segmento; no es una muestra esperada dentro de `[08:00,08:10)`.

In [ ]:
power = telemetry[telemetry.sensor_id == "M02_power"].copy()
power["dt"] = pd.to_datetime(power.timestamp_utc, utc=True)
segments = pd.DataFrame({"inicio": power.dt.iloc[:-1].tolist(), "fin": power.dt.iloc[1:].tolist(), "kW": power.value.iloc[:-1].tolist()})
segments["segundos"] = (segments.fin - segments.inicio).dt.total_seconds()
segments["incluido"] = segments.segundos <= 90
display(segments)
assert segments.loc[segments.incluido, "segundos"].sum() == 420
display(quality[quality.sensor_id == "M02_power"])

**Dos coberturas distintas:** muestras: 8/10 = 80 %. Energía: 7/10 minutos = 70 %. No deben confundirse.

El cociente ingenuo 0,35/8 = 0,04375 parecería mejor que 0,1. Pero divide energía incompleta entre producción completa: no es una comparación válida.

In [ ]:
print("Cociente ingenuo (no usar para comparar):", .35 / 8)
print("Indicador evaluable:", intensidad(.35, 8, .7, 1, False))
print("Con cero piezas:", intensidad(1, 0, 1, 1, False))
assert intensidad(.35, 8, .7, 1, False) is None

**Pregunta para la defensa:** ¿qué información falta para comparar máquinas? Necesitamos energía con cobertura suficiente, producción completa, mismo intervalo y un proceso/producto comparable. Incluso con cobertura completa, producir piezas distintas limita la interpretación industrial.